In [ ]:


import numpy as np
import pandas as pd

PRODUCT_VEC_PATH = "../12_22/final_product_vectors.npy"
PRODUCT_META_PATH = "../12_22/final_product_meta.csv"
BRAND_TONE_PKL_PATH = "../12_22/brand_analysis_result.pkl"

OUT_VEC = "../12_22/final_product_with_brandtone.npy"
OUT_META = "../12_22/final_product_with_brandtone_meta.csv"

product_vecs = np.load(PRODUCT_VEC_PATH)              
product_meta = pd.read_csv(PRODUCT_META_PATH)         
brand_df = pd.read_pickle(BRAND_TONE_PKL_PATH)       

embedding_col = None
for c in brand_df.columns:
    if isinstance(brand_df[c].iloc[0], (list, np.ndarray)):
        embedding_col = c
        break

if embedding_col is None:
    raise ValueError("brand_analysis_result.pkl 에서 임베딩 벡터 컬럼을 찾지 못했습니다.")


BRAND_COL = "브랜드" if "브랜드" in brand_df.columns else "brand"


brand_vec_dict = {
    row[BRAND_COL]: np.array(row[embedding_col], dtype=np.float32)
    for _, row in brand_df.iterrows()
}

brand_vec_dim = len(next(iter(brand_vec_dict.values())))

final_vectors = []
final_meta_rows = []

for i, row in product_meta.iterrows():
    brand = row["brand"] if "brand" in row else row.get("브랜드", None)

    prod_vec = product_vecs[i]

    brand_vec = brand_vec_dict.get(
        brand,
        np.zeros(brand_vec_dim, dtype=np.float32)
    )

    final_vec = np.concatenate([prod_vec, brand_vec], axis=0)
    final_vectors.append(final_vec)

    final_meta_rows.append(row.to_dict())

final_vectors = np.array(final_vectors, dtype=np.float32)

np.save(OUT_VEC, final_vectors)
pd.DataFrame(final_meta_rows).to_csv(OUT_META, index=False)


print("파이프라인 결합 완료")
print("제품 벡터 shape:", product_vecs.shape)
print("브랜드톤 벡터 dim:", brand_vec_dim)
print("최종 벡터 shape:", final_vectors.shape)
print("saved:", OUT_VEC, OUT_META)

파이프라인 결합 완료
제품 벡터 shape: (1581, 1792)
브랜드톤 벡터 dim: 768
최종 벡터 shape: (1581, 2560)
saved: ../12_22/final_product_with_brandtone.npy ../12_22/final_product_with_brandtone_meta.csv
